# Phase 4 : Feature Engineering 

**Objective**

To engineer meaningful and interpretable features from the original survey variables that better represent underlying lifestyle dimensions and improve the performance and interpretability of downstream clustering.

**Outputs**<br>
engineered_df<br>
Feature Engineering Report<br>
Engineered Feature Metadata<br>
<br>
**Assumptions**<br>
Original features are validated.<br>
Original features are preserved.<br>
Every engineered feature must have theoretical justification.<br>

**Expected Deliverables**<br>

By the end of this notebook we should have:<br>

Original features preserved<br>
New engineered features documented<br>
Validation of engineered features<br>
Updated feature metadata

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

engineered_df = pd.read_csv("dataset\\clean_dataset.csv")

# Stastical thinking 

| Category           | Statement                                                                                 |
| ------------------ | ----------------------------------------------------------------------------------------- |
| **FACT**           | Questionnaire items use different response scales.                                        |
| **ASSUMPTION**     | Each questionnaire item should contribute equally to its composite index.                 |
| **HYPOTHESIS**     | Harmonizing scales before combining them produces more balanced engineered features.      |
| **INTERPRETATION** | Normalization removes artificial weighting caused by different score ranges.              |
| **LIMITATION**     | This assumes every questionnaire item has equal importance, which may not always be true. |


In [3]:
rating_scales = {
    "FRUITS_VEGGIES": (0, 5),
    "DAILY_STRESS": (0, 5),
    "PLACES_VISITED": (0, 10),
    "CORE_CIRCLE": (0, 10),
    "SUPPORTING_OTHERS": (0, 10),
    "SOCIAL_NETWORK": (0, 10),
    "ACHIEVEMENT": (0, 10),
    "DONATION": (0, 5),
    "BMI_RANGE": (1, 2),
    "TODO_COMPLETED": (0, 10),
    "FLOW": (0, 10),
    "DAILY_STEPS": (1, 10),
    "LIVE_VISION": (0, 10),
    "SLEEP_HOURS": (1, 10),
    "LOST_VACATION": (0, 10),
    "DAILY_SHOUTING": (0, 10),
    "SUFFICIENT_INCOME": (1, 2),
    "PERSONAL_AWARDS": (0, 10),
    "TIME_FOR_PASSION": (0, 10),
    "WEEKLY_MEDITATION": (0, 10)
}

<h4>Normalizing all the features because somes of them contain a scale of 0-5 whereas some of them contain 0-10 scale</h4>

In [17]:
normalized_df = engineered_df.copy()

# Create normalized feature copies without overwriting originals

for feature, (min_value, max_value) in rating_scales.items():

    normalized_df[f"{feature}_NORM"] = (
        normalized_df[feature] - min_value
    ) / (max_value - min_value)

print("Normalized feature copies created successfully.")

Normalized feature copies created successfully.


we used theoretical scaling instead of min max scaler because :

![proof behind theoretical scaling](./Images/proof_for_theoretical_scaling.png)


<h4>Validating the normalization</h4>

In [18]:
# Create the validation DataFrame by appending _NORM to every raw feature key
validation = pd.DataFrame({
    "Minimum": normalized_df[[f"{feature}_NORM" for feature in rating_scales.keys()]].min(),
    "Maximum": normalized_df[[f"{feature}_NORM" for feature in rating_scales.keys()]].max()
})

display(validation)


,Minimum,Maximum
FRUITS_VEGGIES_NORM,0.0,1.0
DAILY_STRESS_NORM,0.0,1.0
PLACES_VISITED_NORM,0.0,1.0
CORE_CIRCLE_NORM,0.0,1.0
SUPPORTING_OTHERS_NORM,0.0,1.0
SOCIAL_NETWORK_NORM,0.0,1.0
ACHIEVEMENT_NORM,0.0,1.0
DONATION_NORM,0.0,1.0
BMI_RANGE_NORM,0.0,1.0
TODO_COMPLETED_NORM,0.0,1.0


<h4>Now all the features have been verified to have a same scale from 0 - 1</h4>

In [19]:
# Display original and normalized values

comparison_columns = []

for feature in list(rating_scales.keys())[:5]:
    comparison_columns.extend([feature, f"{feature}_NORM"])

display(normalized_df[comparison_columns].head())

,FRUITS_VEGGIES,FRUITS_VEGGIES_NORM,DAILY_STRESS,DAILY_STRESS_NORM,PLACES_VISITED,PLACES_VISITED_NORM,CORE_CIRCLE,CORE_CIRCLE_NORM,SUPPORTING_OTHERS,SUPPORTING_OTHERS_NORM
0,3,0.6,2,0.4,2,0.2,5,0.5,0,0.0
1,2,0.4,3,0.6,4,0.4,3,0.3,8,0.8
2,2,0.4,3,0.6,3,0.3,4,0.4,4,0.4
3,3,0.6,3,0.6,10,1.0,3,0.3,10,1.0
4,5,1.0,1,0.2,3,0.3,3,0.3,10,1.0


# Reverse Coding

In [20]:
# Features requiring reverse coding

reverse_features = [
    "DAILY_STRESS_NORM",
    "LOST_VACATION_NORM",
    "DAILY_SHOUTING_NORM"
]

for feature in reverse_features:

    new_name = feature.replace("_NORM", "_REV")

    normalized_df[new_name] = 1 - normalized_df[feature]

print("Reverse-coded features created successfully.")

Reverse-coded features created successfully.


# Validating reversed features

In [24]:
validation = normalized_df[
    [
        "DAILY_STRESS",
        "DAILY_STRESS_NORM",
        "DAILY_STRESS_REV",
        "LOST_VACATION",
        "LOST_VACATION_NORM",
        "LOST_VACATION_REV"
    ]
].head(10)

display(validation)

,DAILY_STRESS,DAILY_STRESS_NORM,DAILY_STRESS_REV,LOST_VACATION,LOST_VACATION_NORM,LOST_VACATION_REV
0,2,0.4,0.6,5,0.5,0.5
1,3,0.6,0.4,2,0.2,0.8
2,3,0.6,0.4,10,1.0,0.0
3,3,0.6,0.4,7,0.7,0.3
4,1,0.2,0.8,0,0.0,1.0
5,2,0.4,0.6,0,0.0,1.0
6,2,0.4,0.6,10,1.0,0.0
7,4,0.8,0.2,0,0.0,1.0
8,3,0.6,0.4,0,0.0,1.0
9,4,0.8,0.2,0,0.0,1.0
